# Tetrahedral flux reconstruction and adaptive resolution

Canonical Raviart–Thomas reconstruction uses physical flux moments. The error estimator requires $k\geq\ell+d$, hence $k\geq\ell+3$ on tetrahedra. The analytical campaigns distinguish estimator behavior from pressure and vector-flux accuracy.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
import json
from pathlib import Path
import numpy as np
from threadpoolctl import threadpool_limits
from IPython.display import Image, display
from pymhm import TetraMesh
from pymhm.estimators.darcy_3d import estimate_darcy_error_3d

records = root / "examples/results/reconstruction3d"


## Represented nonzero boundary data

The affine exact pressure has constant physical flux $(-1,-2,-3)$. The local $P_3$ space, constant macroface flux and RT1 reconstruction satisfy the three-dimensional estimator hypotheses. Both field errors and all four estimator terms should be at the floating-point solve level.

## Declare local and global forms

The application defines the energy/source and both oriented trace incidences.
Qk Cartesian and Pk tetrahedral coordinates use their shared FEM kernels:

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T, & L_T(v)&=(f,v)_T,\\
b_T(\lambda,v)&=\langle\lambda,v\rangle_{\partial T}, &
c_T(p,\mu)&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The declared `LocalEquations` use C=-B.T, a constant pressure kernel and its
physical integral moment. The additional global `Equation` supplies the weak
Dirichlet pressure functional with its prescribed sign. Neumann data are
outward physical fluxes; a fully Neumann boundary uses an explicit pressure
integral constraint. Generic assembly and field interpretation are separate
calls. The sampled gradient flux remains distinct from the conservative trace.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.tetrahedral_darcy import (
    define_tetrahedral_darcy,
    recover_tetrahedral_darcy,
)


def pressure(x):
    """Affine pressure with exactly represented Dirichlet values."""
    return 1 + x @ np.array([1.0, 2.0, 3.0])


with threadpool_limits(1):
    solution_definition = define_tetrahedral_darcy(
        TetraMesh.unit_cube(), degree=3, dirichlet=pressure
    )
    solution_system = assemble(solution_definition.problem)
    solution = recover_tetrahedral_darcy(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
    estimator = estimate_darcy_error_3d(solution, dirichlet=pressure)
    assert solution.l2_error(pressure) < 1e-11
    assert estimator.reconstruction.flux_l2_error([-1.0, -2.0, -3.0]) < 1e-10
    assert estimator.total < 1e-10
    print("Pressure error:", solution.l2_error(pressure))
    print("Indicator:", estimator.total)


## Uniform and adaptive exact-field studies

The stored research runs use five uniform grids and eight adaptive states. Their error quadrature is checked independently. Effectivity compares the indicator with the broken energy error; it does not replace the physical pressure and flux errors.

In [ ]:
for name in ("uniform", "adaptive"):
    record = json.loads((records / f"{name}.json").read_text())
    assert record["local_degree"] >= record["trace_degree"] + 3
    assert record["source_changed_during_run"] is False
    assert len(record["rows"]) == (5 if name == "uniform" else 8)
    print(name, len(record["rows"]), "states")
    print([(r["macro_cells"], r["pressure_relative"], r["rt_flux_relative"], r["effectivity"]) for r in record["rows"]])
    display(Image(filename=str(root / f"docs/figures/reconstruction3d/{name}-convergence.png")))


## Separate local and skeletal approximation

All rows use the same macrogeometry and exact source. The study varies the local polynomial degree, then the flux trace degree and subface partition. Relative errors use full physical volume norms. In this experiment, $P_4/P_2$ is an approximation-space comparison; it is not assigned the estimator theorem, which would require local $P_5$ for that trace in three dimensions.

In [ ]:
record = json.loads((records / "resolution.json").read_text())
assert record["source_changed_during_run"] is False
for r in record["rows"]:
    print(r["name"], "pressure:", r["pressure_relative"], "raw flux:", r["raw_flux_relative"], "RT flux:", r["rt_flux_relative"])
control = json.loads((records / "reconstruction-order.json").read_text())
assert control["source_changed_during_run"] is False
print("Same pressure solution, RT3 relative flux error:", control["rt_flux_relative"])
for name in ("adaptive-components", "adaptive-profiles", "adaptive-profile-errors"):
    display(Image(filename=str(root / f"docs/figures/reconstruction3d/{name}.png")))
